In [0]:
kpi_datacube = spark.table("novocart_global.novocart_gold.kpi_datacube")
display(kpi_datacube)

In [0]:
from pyspark.sql.functions import sum

kpi_total_revenue = kpi_datacube.filter("order_status = 'completed'").select(
    sum("line_total_usd").alias("total_revenue")
)
display(kpi_total_revenue)

In [0]:
kpi_revenue_country = kpi_datacube \
    .groupBy("country_code") \
    .agg(sum("line_total_usd").alias("revenue"))
display(kpi_revenue_country)

In [0]:
from pyspark.sql.functions import sum

kpi_revenue_channel = kpi_datacube \
    .groupBy("channel") \
    .agg(sum("line_total_usd").alias("revenue"))
display(kpi_revenue_channel)

In [0]:
from pyspark.sql.functions import col

completed_order_count = kpi_datacube \
    .filter(col("order_status") == "completed") \
    .select("order_id") \
    .distinct() \
    .count()
display(spark.createDataFrame([(completed_order_count,)], ["completed_order_count"]))

In [0]:
total_orders = kpi_datacube.select("order_id").distinct().count()
completed_order_rate = completed_order_count / total_orders if total_orders > 0 else None
display(spark.createDataFrame([(completed_order_rate,)], ["completed_order_rate"]))

In [0]:
from pyspark.sql.functions import sum, count

kpi_AOV = kpi_datacube \
    .groupBy("order_id") \
    .agg(sum("line_total_usd").alias("order_total")) \
    .select((sum("order_total") / count("*")).alias("avg_order_value"))
display(kpi_AOV)

In [0]:
from pyspark.sql.functions import sum, col

product_revenue = kpi_datacube \
    .groupBy("product_id", "product_name") \
    .agg(sum("line_total_usd").alias("revenue")) \
    .orderBy(col("revenue").desc()) \
    .limit(5)
display(product_revenue)

In [0]:
kpi_active_customers = kpi_datacube \
    .select("customer_id") \
    .distinct() \
    .count()
display(spark.createDataFrame([(kpi_active_customers,)], ["active_customers_count"]))

In [0]:
from pyspark.sql.functions import year, month

kpi_customer_acquisition = kpi_datacube \
    .withColumn("year", year("registration_date")) \
    .withColumn("month", month("registration_date")) \
    .select("customer_id", "year", "month") \
    .distinct() \
    .groupBy("year", "month") \
    .count()
display(kpi_customer_acquisition)

In [0]:
from pyspark.sql.functions import col

total_records = kpi_datacube.count()
valid_records = kpi_datacube \
    .filter(col("customer_name") != "Unknown Customer") \
    .count()
data_quality_score = valid_records / total_records if total_records > 0 else None
display(spark.createDataFrame([(data_quality_score,)], ["data_quality_score"]))

